# What the suite covers, and how it is driven

## How the tests are driven

There is **no pytest configuration** — no `pytest.ini`, no `[tool.pytest.ini_options]`.
Discovery is entirely the default: `pytest tests` walks the tree, imports every
`test_*.py` and runs every `test_*` function. Nothing opts a directory in or
out, so a new `test_*.py` anywhere under `tests/` is picked up the moment it
exists.

Two things shape that:

- **`tests/conftest.py`** puts the repository root and every directory holding
  a `_setup_*.py` onto `sys.path`, so a test can import its helper from
  wherever it sits.
- **No test directory is a package.** `02_basetest` starts with a digit and
  `pptx-basic` has a hyphen, so neither is a valid identifier. Every test
  module is therefore imported *top-level*, which is why each shared helper
  needs a name unique across the whole suite — `_setup_rdf`, `_setup_values`,
  and so on. Two helpers with one name silently shadow each other.

`tests/baseTestRoot.py` supplies the autouse `reset_state` fixture, which
clears the parser's process-global state between tests.

## Two layers of coverage per case

A case directory such as `04_examples/wordreport` is covered twice, and the two
answer different questions:

| | asks | catches |
|---|---|---|
| `test_docx_generation.py` | did a document come out? | crashes, empty output |
| `differential/test_rdf_yaml_equivalence.py` | does it **say the right thing**? | wrong content, wrong order, missing blocks |

The first asserts a file exists and is non-empty. That is a real smoke test and
it is also why the clone-ordering defect survived for years with a green suite.
The second builds the document and compares its text against
`<case>/expected/<stem>.json`, the reference captured from the old `.rdf`.

**When a case moves to YAML**, both need pointing at the new document — that is
what `rdf_name="word_input.yaml"` in the case config now does. `rdf_name` keeps
its name because the thing keeps its name: it is still the report data file.


In [ ]:
import subprocess
import sys
from pathlib import Path

TESTS = Path.cwd() if Path.cwd().name == 'tests' else Path.cwd() / 'tests'
WORKTREE = TESTS.parent

print('python  ', sys.executable)
print('worktree', WORKTREE)
print('branch  ', subprocess.run(['git', 'branch', '--show-current'], cwd=WORKTREE,
                                 capture_output=True, text=True).stdout.strip())
assert (TESTS / 'conftest.py').is_file(), f'not a tests directory: {TESTS}'


## Run the suite and collect the results

pytest runs in a **subprocess** on purpose. Its own process-global reset is
per-test, but this notebook's kernel is long-lived and re-running in-process
would let one run's imports and class attributes leak into the next. A
subprocess also means this cell can be re-run as often as you like.

The results come back as JUnit XML, which is structured — no parsing of
terminal output.


In [ ]:
import tempfile
import xml.etree.ElementTree as ET
from collections import Counter

def run_suite(*extra, target='.'):
    """Run pytest and return one record per test."""
    report = Path(tempfile.mkdtemp()) / 'report.xml'
    finished = subprocess.run(
        [sys.executable, '-m', 'pytest', target, '-q', '--tb=no',
         f'--junitxml={report}', *extra],
        cwd=TESTS, capture_output=True, text=True)

    if not report.is_file():
        print(finished.stdout[-3000:])
        raise SystemExit('pytest did not produce a report')

    records = []
    for case in ET.parse(report).getroot().iter('testcase'):
        outcome = 'passed'
        message = ''
        for child in case:
            if child.tag in ('failure', 'error'):
                outcome, message = child.tag, (child.get('message') or '')
            elif child.tag == 'skipped':
                outcome = 'xfailed' if 'xfail' in (child.get('type') or '') else 'skipped'
                message = child.get('message') or ''
        records.append({
            'file': case.get('classname', '').replace('.', '/') + '.py',
            'name': case.get('name'),
            'time': float(case.get('time', 0)),
            'outcome': outcome,
            'message': message.strip().split(chr(10))[0][:160],
        })
    return records

RESULTS = run_suite()
print(len(RESULTS), 'tests')
print(dict(Counter(r['outcome'] for r in RESULTS)))


## Where the tests live

Grouped by directory, so it is obvious which areas are thin. `time` is the
wall clock the group cost — the docx cases dominate, because they really do
build documents.


In [ ]:
def by(records, key):
    groups = {}
    for record in records:
        groups.setdefault(key(record), []).append(record)
    return groups

def table(groups, title):
    width = max(len(name) for name in groups)
    print(f'{title:<{width}}  {"tests":>5} {"pass":>5} {"other":>6} {"time":>7}')
    print('-' * (width + 27))
    for name in sorted(groups):
        rows = groups[name]
        passed = sum(1 for r in rows if r['outcome'] == 'passed')
        other = len(rows) - passed
        print(f'{name:<{width}}  {len(rows):>5} {passed:>5} '
              f'{other if other else "-":>6} {sum(r["time"] for r in rows):>6.2f}s')
    print('-' * (width + 27))
    print(f'{"total":<{width}}  {len(records_total):>5} '
          f'{sum(1 for r in records_total if r["outcome"] == "passed"):>5}')

records_total = RESULTS
table(by(RESULTS, lambda r: str(Path(r['file']).parent)), 'directory')


In [ ]:
table(by(RESULTS, lambda r: r['file']), 'module')


## Anything not simply passing

An `xfailed` here is a known gap with a written reason, not a mystery — the
reason is the marker's, and it is the thing to read before touching that area.


In [ ]:
noteworthy = [r for r in RESULTS if r['outcome'] != 'passed']
if not noteworthy:
    print('everything passed')
for record in noteworthy:
    print(f'[{record["outcome"]}] {record["file"]}::{record["name"]}')
    if record['message']:
        print(f'         {record["message"]}')


## What covers each case directory

A case is a directory holding a document, a template and its data. This shows
which of the two layers each one has — a case with fixtures but no reference is
only smoke-tested, which is worth knowing before trusting it.


In [ ]:
def case_directories():
    seen = {}
    for path in sorted(TESTS.rglob('*.yaml')) + sorted(TESTS.rglob('*.rdf')):
        if 'expected' in path.parts:
            continue
        seen.setdefault(path.parent, {'yaml': 0, 'rdf': 0})[path.suffix[1:]] += 1
    return seen

rows = []
for directory, counts in sorted(case_directories().items()):
    relative = directory.relative_to(TESTS)
    templates = sorted(p.suffix for p in directory.glob('*.doc*')) + \
                sorted(p.suffix for p in directory.glob('*.ppt*'))
    rows.append((
        str(relative),
        counts['yaml'],
        counts['rdf'],
        'yes' if (directory / 'expected').is_dir() else '-',
        'yes' if any(directory.glob('test_*.py')) else '-',
        'yes' if templates else '-',
    ))

width = max(len(r[0]) for r in rows)
print(f'{"case":<{width}}  {"yaml":>4} {"rdf":>4} {"reference":>9} '
      f'{"own test":>8} {"template":>8}')
print('-' * (width + 40))
for row in rows:
    print(f'{row[0]:<{width}}  {row[1]:>4} {row[2]:>4} {row[3]:>9} '
          f'{row[4]:>8} {row[5]:>8}')


## Run a slice

`-k` selects by name, a path selects by place. Both are just pytest, so
anything you can type on the command line works here.


In [ ]:
subset = run_suite('-k', 'clone or differential')
for record in subset:
    print(f'[{record["outcome"]}] {record["name"]}')


In [ ]:
subset = run_suite(target='04_examples/wordreport')
for record in subset:
    print(f'[{record["outcome"]}] {record["file"]}::{record["name"]}')


## The one deliberate gap

`word_text` is `xfail(strict=True)`, and it is a **template** problem rather
than a loader one: `template_text.docx` spells its depth-3 block
`<subsubsubsection:secondsubsubi>`, which is not in the docx ladder — every
other template in the corpus stops at `subsubsection`. The `.yaml` uses the
ladder's name, `sub3section`, so that template needs the matching tag rename.

Strict on purpose: the day someone renames the tag, the test **fails** for
passing, which is how the change announces itself.


In [ ]:
for record in RESULTS:
    if record['outcome'] == 'xfailed':
        print(record['file'])
        print(' ', record['name'])
        print(' ', record['message'][:300])
